# BikeToDrive – Data Warehouse Implementation

Dit notebook maakt het Data Warehouse in SQLite en implementeert de ETL in Python.

Keuzes in deze uitwerking:
- **3 feiten**: `fact_verkoop`, `fact_inkoop`, `fact_onderhoud`
- **SCD Type 2**: `dim_klant` en `dim_monteur`
- **SCD Type 1**: alle overige dimensies
- **Inlaadstrategie**: incrementeel laden op basis van business keys; nieuwe feitregels worden toegevoegd, dimensies worden bijgewerkt via Type 1 of Type 2

In [1]:

import sqlite3
import pandas as pd
import os
import shutil
from pathlib import Path

BASE_DIR = Path.cwd()
SOURCE_DIR = BASE_DIR
WORK_DIR = BASE_DIR / "working_sources"
DWH_PATH = BASE_DIR / "BikeToDrive_DWH.db"

ORIGINAL_SOURCES = {
    'acc_sale': SOURCE_DIR / 'BikeToDrive_1_Accessoireverkoop.db',
    'bike_sale': SOURCE_DIR / 'BikeToDrive_2_Fietsverkoop.db',
    'maint': SOURCE_DIR / 'BikeToDrive_3_Onderhoud.db',
    'acc_buy': SOURCE_DIR / 'BikeToDrive_4_Accessoire_Inkoop.db',
    'bike_buy': SOURCE_DIR / 'BikeToDrive_5_Fiets_Inkoop.db',
}

WORK_SOURCES = {k: WORK_DIR / v.name for k, v in ORIGINAL_SOURCES.items()}

SCHEMA_SQL = """PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL UNIQUE,
    day_of_month INTEGER NOT NULL,
    month_num INTEGER NOT NULL,
    month_name TEXT NOT NULL,
    quarter_num INTEGER NOT NULL,
    year_num INTEGER NOT NULL,
    week_num INTEGER NOT NULL,
    day_name TEXT NOT NULL,
    is_weekend INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS dim_filiaal (
    filiaal_key INTEGER PRIMARY KEY AUTOINCREMENT,
    filiaal_bk INTEGER NOT NULL UNIQUE,
    naam TEXT,
    adres TEXT,
    provincie TEXT
);

CREATE TABLE IF NOT EXISTS dim_klant (
    klant_key INTEGER PRIMARY KEY AUTOINCREMENT,
    klant_bk INTEGER NOT NULL,
    naam TEXT,
    adres TEXT,
    woonplaats TEXT,
    geslacht TEXT,
    geboortedatum TEXT,
    leeftijd INTEGER,
    leeftijdsklasse TEXT,
    valid_from TEXT NOT NULL,
    valid_to TEXT,
    is_current INTEGER NOT NULL DEFAULT 1,
    UNIQUE (klant_bk, valid_from)
);

CREATE TABLE IF NOT EXISTS dim_monteur (
    monteur_key INTEGER PRIMARY KEY AUTOINCREMENT,
    monteur_bk INTEGER NOT NULL,
    naam TEXT,
    woonplaats TEXT,
    uurloon REAL,
    looncategorie TEXT,
    filiaal_bk INTEGER,
    valid_from TEXT NOT NULL,
    valid_to TEXT,
    is_current INTEGER NOT NULL DEFAULT 1,
    UNIQUE (monteur_bk, valid_from)
);

CREATE TABLE IF NOT EXISTS dim_leverancier (
    leverancier_key INTEGER PRIMARY KEY AUTOINCREMENT,
    leverancier_bk INTEGER NOT NULL UNIQUE,
    naam TEXT,
    adres TEXT,
    woonplaats TEXT
);

CREATE TABLE IF NOT EXISTS dim_fabrikant (
    fabrikant_key INTEGER PRIMARY KEY AUTOINCREMENT,
    fabrikant_bk INTEGER NOT NULL UNIQUE,
    naam TEXT,
    adres TEXT,
    plaats TEXT
);

CREATE TABLE IF NOT EXISTS dim_product (
    product_key INTEGER PRIMARY KEY AUTOINCREMENT,
    product_bk TEXT NOT NULL UNIQUE,
    product_type TEXT NOT NULL,
    source_domain TEXT NOT NULL,
    natural_key INTEGER NOT NULL,
    naam TEXT,
    soort TEXT,
    merk TEXT,
    model_type TEXT,
    kleur TEXT,
    standaardprijs REAL,
    inkoopprijs REAL,
    prijssegment TEXT,
    winstmarge_pct REAL,
    leverancier_bk INTEGER,
    fabrikant_bk INTEGER
);

CREATE TABLE IF NOT EXISTS fact_verkoop (
    verkoop_key INTEGER PRIMARY KEY AUTOINCREMENT,
    verkoop_bk TEXT NOT NULL UNIQUE,
    date_key INTEGER NOT NULL,
    klant_key INTEGER NOT NULL,
    monteur_key INTEGER NOT NULL,
    filiaal_key INTEGER NOT NULL,
    product_key INTEGER NOT NULL,
    verkoop_type TEXT NOT NULL,
    aantal INTEGER NOT NULL,
    verkoopprijs_eenheid REAL NOT NULL,
    omzet_bedrag REAL NOT NULL,
    kostprijs_eenheid REAL,
    brutowinst_bedrag REAL,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (klant_key) REFERENCES dim_klant(klant_key),
    FOREIGN KEY (monteur_key) REFERENCES dim_monteur(monteur_key),
    FOREIGN KEY (filiaal_key) REFERENCES dim_filiaal(filiaal_key),
    FOREIGN KEY (product_key) REFERENCES dim_product(product_key)
);

CREATE TABLE IF NOT EXISTS fact_inkoop (
    inkoop_key INTEGER PRIMARY KEY AUTOINCREMENT,
    inkoop_bk TEXT NOT NULL UNIQUE,
    date_key INTEGER NOT NULL,
    product_key INTEGER NOT NULL,
    leverancier_key INTEGER,
    fabrikant_key INTEGER,
    inkoop_type TEXT NOT NULL,
    aantal INTEGER NOT NULL,
    inkoopprijs_eenheid REAL,
    inkoopbedrag REAL,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (product_key) REFERENCES dim_product(product_key),
    FOREIGN KEY (leverancier_key) REFERENCES dim_leverancier(leverancier_key),
    FOREIGN KEY (fabrikant_key) REFERENCES dim_fabrikant(fabrikant_key)
);

CREATE TABLE IF NOT EXISTS fact_onderhoud (
    onderhoud_key INTEGER PRIMARY KEY AUTOINCREMENT,
    onderhoud_bk TEXT NOT NULL UNIQUE,
    date_key INTEGER NOT NULL,
    monteur_key INTEGER NOT NULL,
    filiaal_key INTEGER NOT NULL,
    product_key INTEGER NOT NULL,
    starttijd TEXT,
    eindtijd TEXT,
    duur_uren REAL,
    uurloon REAL,
    arbeidskosten REAL,
    FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
    FOREIGN KEY (monteur_key) REFERENCES dim_monteur(monteur_key),
    FOREIGN KEY (filiaal_key) REFERENCES dim_filiaal(filiaal_key),
    FOREIGN KEY (product_key) REFERENCES dim_product(product_key)
);"""

def connect(db_path, foreign_keys=True):
    con = sqlite3.connect(db_path)
    con.execute(f"PRAGMA foreign_keys = {'ON' if foreign_keys else 'OFF'}")
    return con

def reset_working_sources():
    WORK_DIR.mkdir(exist_ok=True)
    for source in WORK_SOURCES.values():
        if source.exists():
            source.unlink()
    for key, src in ORIGINAL_SOURCES.items():
        shutil.copy2(src, WORK_SOURCES[key])

def reset_dwh():
    if DWH_PATH.exists():
        DWH_PATH.unlink()
    con = connect(DWH_PATH)
    con.executescript(SCHEMA_SQL)
    con.commit()
    con.close()

def read_table(db_path, table_name):
    con = connect(db_path)
    df = pd.read_sql_query(f'SELECT * FROM "{table_name}"', con)
    con.close()
    return df

def build_staging(src_paths):
    klant = pd.concat([
        read_table(src_paths['acc_sale'], 'Klant'),
        read_table(src_paths['bike_sale'], 'Klant')
    ], ignore_index=True).drop_duplicates(subset=['klantnr']).sort_values('klantnr')
    klant['geboortedatum'] = pd.to_datetime(klant['geboortedatum'])
    today = pd.Timestamp('2026-04-08')
    klant['leeftijd'] = ((today - klant['geboortedatum']).dt.days / 365.25).astype(int)
    klant['leeftijdsklasse'] = pd.cut(klant['leeftijd'], bins=[-1, 24, 39, 59, 200], labels=['<25', '25-39', '40-59', '60+'])
    klant['geboortedatum'] = klant['geboortedatum'].dt.strftime('%Y-%m-%d')

    filiaal = pd.concat([
        read_table(src_paths['acc_sale'], 'Filiaal'),
        read_table(src_paths['bike_sale'], 'Filiaal'),
        read_table(src_paths['maint'], 'Filiaal')
    ], ignore_index=True).drop_duplicates(subset=['filiaalnr']).sort_values('filiaalnr')

    monteur = pd.concat([
        read_table(src_paths['acc_sale'], 'Monteur'),
        read_table(src_paths['bike_sale'], 'Monteur'),
        read_table(src_paths['maint'], 'Monteur')
    ], ignore_index=True).drop_duplicates(subset=['monteurnr']).sort_values('monteurnr')
    monteur['looncategorie'] = pd.cut(monteur['uurloon'], bins=[-1, 18.5, 20, 999], labels=['Laag', 'Midden', 'Hoog'])

    leverancier = pd.concat([
        read_table(src_paths['acc_sale'], 'Leverancier'),
        read_table(src_paths['acc_buy'], 'Leverancier')
    ], ignore_index=True).drop_duplicates(subset=['leveranciernr']).sort_values('leveranciernr')

    fabrikant = pd.concat([
        read_table(src_paths['bike_sale'], 'Fabrikant'),
        read_table(src_paths['maint'], 'Fabrikant'),
        read_table(src_paths['bike_buy'], 'Fabrikant')
    ], ignore_index=True).drop_duplicates(subset=['fabrikantnr']).sort_values('fabrikantnr')

    acc_products = pd.concat([
        read_table(src_paths['acc_sale'], 'Accessoire').assign(source_domain='VERKOOP'),
        read_table(src_paths['acc_buy'], 'Accessoire').assign(source_domain='INKOOP')
    ], ignore_index=True)
    acc_products['product_bk'] = acc_products.apply(lambda r: f"ACCESSOIRE_{r['source_domain']}_{int(r['accessoirenr'])}", axis=1)
    acc_products['product_type'] = 'ACCESSOIRE'
    acc_products['natural_key'] = acc_products['accessoirenr']
    acc_products['prijssegment'] = pd.cut(acc_products['standaardprijs'], bins=[-1, 15, 30, 99999], labels=['Laag', 'Midden', 'Hoog'])
    acc_products['winstmarge_pct'] = ((acc_products['standaardprijs'] - acc_products['inkoopprijs']) / acc_products['standaardprijs'] * 100).round(2)

    bike_products = pd.concat([
        read_table(src_paths['bike_sale'], 'Fiets').assign(source_domain='VERKOOP'),
        read_table(src_paths['maint'], 'Fiets').assign(source_domain='ONDERHOUD'),
        read_table(src_paths['bike_buy'], 'Fiets').assign(source_domain='INKOOP')
    ], ignore_index=True)
    bike_products['product_bk'] = bike_products.apply(lambda r: f"FIETS_{r['source_domain']}_{int(r['fietsnr'])}", axis=1)
    bike_products['product_type'] = 'FIETS'
    bike_products['natural_key'] = bike_products['fietsnr']
    bike_products['naam'] = bike_products['merk'] + ' ' + bike_products['type']
    bike_products['prijssegment'] = pd.cut(bike_products['standaardprijs'], bins=[-1, 700, 1000, 999999], labels=['Laag', 'Midden', 'Hoog'])
    bike_products['winstmarge_pct'] = ((bike_products['standaardprijs'] - bike_products['inkoopprijs']) / bike_products['standaardprijs'] * 100).round(2)

    product = pd.concat([
        acc_products[['product_bk', 'product_type', 'source_domain', 'natural_key', 'naam', 'soort', 'standaardprijs', 'inkoopprijs', 'prijssegment', 'winstmarge_pct', 'leverancier']]
            .rename(columns={'leverancier': 'leverancier_bk'})
            .assign(merk=None, model_type=None, kleur=None, fabrikant_bk=None),
        bike_products[['product_bk', 'product_type', 'source_domain', 'natural_key', 'naam', 'soort', 'merk', 'type', 'kleur', 'standaardprijs', 'inkoopprijs', 'prijssegment', 'winstmarge_pct', 'fabrikant']]
            .rename(columns={'type': 'model_type', 'fabrikant': 'fabrikant_bk'})
            .assign(leverancier_bk=None)
    ], ignore_index=True)

    dates = []
    dates.extend(pd.to_datetime(read_table(src_paths['acc_sale'], 'Accessoire_Verkoop')['datum']).dt.date.astype(str).tolist())
    dates.extend(pd.to_datetime(read_table(src_paths['bike_sale'], 'Fiets_Verkoop')['datum']).dt.date.astype(str).tolist())
    dates.extend(pd.to_datetime(read_table(src_paths['maint'], 'Onderhoud')['datum']).dt.date.astype(str).tolist())
    for tbl, path in [('Accessoire_Inkoop', src_paths['acc_buy']), ('Fiets_Inkoop', src_paths['bike_buy'])]:
        df = read_table(path, tbl)
        for _, row in df.iterrows():
            dates.append(f"{int(row['inkoopjaar']):04d}-{int(row['inkoopmaand']):02d}-01")

    date_df = pd.DataFrame({'full_date': sorted(set(dates))})
    dts = pd.to_datetime(date_df['full_date'])
    date_df['date_key'] = dts.dt.strftime('%Y%m%d').astype(int)
    date_df['day_of_month'] = dts.dt.day
    date_df['month_num'] = dts.dt.month
    date_df['month_name'] = dts.dt.month_name()
    date_df['quarter_num'] = dts.dt.quarter
    date_df['year_num'] = dts.dt.year
    date_df['week_num'] = dts.dt.isocalendar().week.astype(int)
    date_df['day_name'] = dts.dt.day_name()
    date_df['is_weekend'] = dts.dt.dayofweek.isin([5, 6]).astype(int)

    acc_sale = read_table(src_paths['acc_sale'], 'Accessoire_Verkoop')
    acc_sale['verkoop_bk'] = acc_sale['accessoire_verkoopnr'].map(lambda x: f"ACCESSOIRE_{int(x)}")
    acc_sale['verkoop_type'] = 'ACCESSOIRE'
    acc_sale['product_bk'] = acc_sale['accessoire'].map(lambda x: f"ACCESSOIRE_VERKOOP_{int(x)}")
    acc_sale['date_key'] = pd.to_datetime(acc_sale['datum']).dt.strftime('%Y%m%d').astype(int)
    acc_sale = acc_sale.rename(columns={'klant': 'klant_bk', 'monteur': 'monteur_bk', 'verkoopprijs': 'verkoopprijs_eenheid'})
    acc_sale = acc_sale.merge(product[['product_bk', 'inkoopprijs']], on='product_bk', how='left')
    acc_sale['omzet_bedrag'] = (acc_sale['aantal'] * acc_sale['verkoopprijs_eenheid']).round(2)
    acc_sale['kostprijs_eenheid'] = acc_sale['inkoopprijs']
    acc_sale['brutowinst_bedrag'] = (acc_sale['omzet_bedrag'] - acc_sale['aantal'] * acc_sale['kostprijs_eenheid']).round(2)

    bike_sale = read_table(src_paths['bike_sale'], 'Fiets_Verkoop')
    bike_sale['verkoop_bk'] = bike_sale['fiets_verkoopnr'].map(lambda x: f"FIETS_{int(x)}")
    bike_sale['verkoop_type'] = 'FIETS'
    bike_sale['product_bk'] = bike_sale['fiets'].map(lambda x: f"FIETS_VERKOOP_{int(x)}")
    bike_sale['date_key'] = pd.to_datetime(bike_sale['datum']).dt.strftime('%Y%m%d').astype(int)
    bike_sale = bike_sale.rename(columns={'klant': 'klant_bk', 'monteur': 'monteur_bk', 'verkoopprijs': 'verkoopprijs_eenheid'})
    bike_sale = bike_sale.merge(product[['product_bk', 'inkoopprijs']], on='product_bk', how='left')
    bike_sale['omzet_bedrag'] = (bike_sale['aantal'] * bike_sale['verkoopprijs_eenheid']).round(2)
    bike_sale['kostprijs_eenheid'] = bike_sale['inkoopprijs']
    bike_sale['brutowinst_bedrag'] = (bike_sale['omzet_bedrag'] - bike_sale['aantal'] * bike_sale['kostprijs_eenheid']).round(2)

    verkoop = pd.concat([
        acc_sale[['verkoop_bk', 'date_key', 'klant_bk', 'monteur_bk', 'product_bk', 'verkoop_type', 'aantal', 'verkoopprijs_eenheid', 'omzet_bedrag', 'kostprijs_eenheid', 'brutowinst_bedrag']],
        bike_sale[['verkoop_bk', 'date_key', 'klant_bk', 'monteur_bk', 'product_bk', 'verkoop_type', 'aantal', 'verkoopprijs_eenheid', 'omzet_bedrag', 'kostprijs_eenheid', 'brutowinst_bedrag']]
    ], ignore_index=True)

    acc_buy = read_table(src_paths['acc_buy'], 'Accessoire_Inkoop')
    acc_buy['inkoop_bk'] = acc_buy['inkoopnr'].map(lambda x: f"ACCESSOIRE_{int(x)}")
    acc_buy['inkoop_type'] = 'ACCESSOIRE'
    acc_buy['product_bk'] = acc_buy['accessoire'].map(lambda x: f"ACCESSOIRE_INKOOP_{int(x)}")
    acc_buy['date_key'] = acc_buy.apply(lambda r: int(f"{int(r['inkoopjaar']):04d}{int(r['inkoopmaand']):02d}01"), axis=1)
    acc_buy = acc_buy.merge(product[['product_bk', 'inkoopprijs', 'leverancier_bk']], on='product_bk', how='left')
    acc_buy['inkoopprijs_eenheid'] = acc_buy['inkoopprijs']
    acc_buy['inkoopbedrag'] = (acc_buy['aantal'] * acc_buy['inkoopprijs_eenheid']).round(2)

    bike_buy = read_table(src_paths['bike_buy'], 'Fiets_Inkoop')
    bike_buy['inkoop_bk'] = bike_buy['inkoopnr'].map(lambda x: f"FIETS_{int(x)}")
    bike_buy['inkoop_type'] = 'FIETS'
    bike_buy['product_bk'] = bike_buy['fiets'].map(lambda x: f"FIETS_INKOOP_{int(x)}")
    bike_buy['date_key'] = bike_buy.apply(lambda r: int(f"{int(r['inkoopjaar']):04d}{int(r['inkoopmaand']):02d}01"), axis=1)
    bike_buy = bike_buy.merge(product[['product_bk', 'inkoopprijs', 'fabrikant_bk']], on='product_bk', how='left')
    bike_buy['inkoopprijs_eenheid'] = bike_buy['inkoopprijs']
    bike_buy['inkoopbedrag'] = (bike_buy['aantal'] * bike_buy['inkoopprijs_eenheid']).round(2)

    inkoop = pd.concat([
        acc_buy[['inkoop_bk', 'date_key', 'product_bk', 'leverancier_bk', 'inkoop_type', 'aantal', 'inkoopprijs_eenheid', 'inkoopbedrag']].assign(fabrikant_bk=None),
        bike_buy[['inkoop_bk', 'date_key', 'product_bk', 'fabrikant_bk', 'inkoop_type', 'aantal', 'inkoopprijs_eenheid', 'inkoopbedrag']].assign(leverancier_bk=None)
    ], ignore_index=True)

    maint = read_table(src_paths['maint'], 'Onderhoud')
    maint['onderhoud_bk'] = maint['onderhoudnr'].map(lambda x: f"ONDERHOUD_{int(x)}")
    maint['product_bk'] = maint['fiets'].map(lambda x: f"FIETS_ONDERHOUD_{int(x)}")
    maint['date_key'] = pd.to_datetime(maint['datum']).dt.strftime('%Y%m%d').astype(int)
    maint['start_dt'] = pd.to_datetime(maint['datum'] + ' ' + maint['starttijd'].str.slice(0, 8))
    maint['end_dt'] = pd.to_datetime(maint['datum'] + ' ' + maint['eindtijd'].str.slice(0, 8))
    maint['duur_uren'] = ((maint['end_dt'] - maint['start_dt']).dt.total_seconds() / 3600).round(2)
    maint = maint.rename(columns={'monteur': 'monteur_bk'})
    maint = maint.merge(monteur[['monteurnr', 'uurloon']], left_on='monteur_bk', right_on='monteurnr', how='left')
    maint['arbeidskosten'] = (maint['duur_uren'] * maint['uurloon']).round(2)
    onderhoud = maint[['onderhoud_bk', 'date_key', 'monteur_bk', 'product_bk', 'starttijd', 'eindtijd', 'duur_uren', 'uurloon', 'arbeidskosten']]

    return {
        'dim_date': date_df,
        'dim_filiaal': filiaal,
        'dim_klant': klant,
        'dim_monteur': monteur,
        'dim_leverancier': leverancier,
        'dim_fabrikant': fabrikant,
        'dim_product': product,
        'fact_verkoop': verkoop,
        'fact_inkoop': inkoop,
        'fact_onderhoud': onderhoud,
    }

def table_exists(con, name):
    return con.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name=?", (name,)).fetchone() is not None

def load_dim_type1(con, table, df, bk_col, col_map):
    cur = con.cursor()
    for _, row in df.iterrows():
        bk_value = row[bk_col]
        exists = cur.execute(f"SELECT 1 FROM {table} WHERE {col_map[bk_col]} = ?", (bk_value,)).fetchone()
        if exists:
            update_cols = [target for source, target in col_map.items() if source != bk_col]
            update_values = [None if pd.isna(row[source]) else row[source] for source, target in col_map.items() if source != bk_col]
            update_values.append(bk_value)
            cur.execute(
                f"UPDATE {table} SET " + ", ".join([f"{col} = ?" for col in update_cols]) + f" WHERE {col_map[bk_col]} = ?",
                update_values
            )
        else:
            target_cols = list(col_map.values())
            insert_values = [None if pd.isna(row[source]) else row[source] for source in col_map.keys()]
            cur.execute(
                f"INSERT INTO {table} ({', '.join(target_cols)}) VALUES ({', '.join(['?'] * len(target_cols))})",
                insert_values
            )
    con.commit()

def load_dim_type2(con, table, df, bk_col, tracked_src_cols, col_map, load_date='2026-04-08'):
    cur = con.cursor()
    existing = pd.read_sql_query(f"SELECT * FROM {table}", con)
    bk_target = col_map[bk_col]
    for _, row in df.iterrows():
        current = existing[(existing[bk_target] == row[bk_col]) & (existing['is_current'] == 1)]
        new_values = {col_map[source]: (None if pd.isna(row[source]) else row[source]) for source in col_map}
        if current.empty:
            cols = list(new_values.keys()) + ['valid_from', 'valid_to', 'is_current']
            cur.execute(
                f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join(['?'] * len(cols))})",
                list(new_values.values()) + [load_date, None, 1]
            )
            existing = pd.read_sql_query(f"SELECT * FROM {table}", con)
            continue

        current_row = current.iloc[0]
        changed = False
        for src_col in tracked_src_cols:
            target_col = col_map[src_col]
            old = current_row[target_col]
            new = new_values[target_col]
            if (pd.isna(old) and pd.isna(new)) or (old == new):
                continue
            changed = True
            break

        if changed:
            cur.execute(
                f"UPDATE {table} SET valid_to = ?, is_current = 0 WHERE {bk_target} = ? AND is_current = 1",
                (load_date, row[bk_col])
            )
            cols = list(new_values.keys()) + ['valid_from', 'valid_to', 'is_current']
            cur.execute(
                f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join(['?'] * len(cols))})",
                list(new_values.values()) + [load_date, None, 1]
            )
            existing = pd.read_sql_query(f"SELECT * FROM {table}", con)
    con.commit()

def sync_date_dim(con, df):
    existing = set(pd.read_sql_query("SELECT date_key FROM dim_date", con)['date_key'].tolist())
    new_rows = df[~df['date_key'].isin(existing)]
    if not new_rows.empty:
        new_rows.to_sql('dim_date', con, if_exists='append', index=False)

def load_facts(con, staging):
    filiaal_lookup = pd.read_sql_query("SELECT filiaal_key, filiaal_bk FROM dim_filiaal", con).set_index('filiaal_bk')['filiaal_key'].to_dict()
    product_lookup = pd.read_sql_query("SELECT product_key, product_bk FROM dim_product", con).set_index('product_bk')['product_key'].to_dict()
    leverancier_lookup = pd.read_sql_query("SELECT leverancier_key, leverancier_bk FROM dim_leverancier", con).set_index('leverancier_bk')['leverancier_key'].to_dict()
    fabrikant_lookup = pd.read_sql_query("SELECT fabrikant_key, fabrikant_bk FROM dim_fabrikant", con).set_index('fabrikant_bk')['fabrikant_key'].to_dict()
    klant_lookup = pd.read_sql_query("SELECT klant_key, klant_bk FROM dim_klant WHERE is_current = 1", con).set_index('klant_bk')['klant_key'].to_dict()
    monteur_current = pd.read_sql_query("SELECT monteur_key, monteur_bk, filiaal_bk, uurloon FROM dim_monteur WHERE is_current = 1", con)
    monteur_lookup = monteur_current.set_index('monteur_bk')['monteur_key'].to_dict()
    monteur_filiaal = monteur_current.set_index('monteur_bk')['filiaal_bk'].to_dict()
    monteur_uurloon = monteur_current.set_index('monteur_bk')['uurloon'].to_dict()

    existing_verkoop = set(pd.read_sql_query("SELECT verkoop_bk FROM fact_verkoop", con)['verkoop_bk'].tolist())
    verkoop_rows = []
    for _, row in staging['fact_verkoop'].iterrows():
        if row['verkoop_bk'] in existing_verkoop:
            continue
        filiaal_bk = monteur_filiaal.get(row['monteur_bk'])
        verkoop_rows.append({
            'verkoop_bk': row['verkoop_bk'],
            'date_key': int(row['date_key']),
            'klant_key': klant_lookup.get(row['klant_bk']),
            'monteur_key': monteur_lookup.get(row['monteur_bk']),
            'filiaal_key': filiaal_lookup.get(filiaal_bk),
            'product_key': product_lookup.get(row['product_bk']),
            'verkoop_type': row['verkoop_type'],
            'aantal': int(row['aantal']),
            'verkoopprijs_eenheid': float(row['verkoopprijs_eenheid']),
            'omzet_bedrag': float(row['omzet_bedrag']),
            'kostprijs_eenheid': None if pd.isna(row['kostprijs_eenheid']) else float(row['kostprijs_eenheid']),
            'brutowinst_bedrag': None if pd.isna(row['brutowinst_bedrag']) else float(row['brutowinst_bedrag'])
        })
    if verkoop_rows:
        pd.DataFrame(verkoop_rows).to_sql('fact_verkoop', con, if_exists='append', index=False)

    existing_inkoop = set(pd.read_sql_query("SELECT inkoop_bk FROM fact_inkoop", con)['inkoop_bk'].tolist())
    inkoop_rows = []
    for _, row in staging['fact_inkoop'].iterrows():
        if row['inkoop_bk'] in existing_inkoop:
            continue
        inkoop_rows.append({
            'inkoop_bk': row['inkoop_bk'],
            'date_key': int(row['date_key']),
            'product_key': product_lookup.get(row['product_bk']),
            'leverancier_key': None if pd.isna(row.get('leverancier_bk')) else leverancier_lookup.get(int(row['leverancier_bk'])),
            'fabrikant_key': None if pd.isna(row.get('fabrikant_bk')) else fabrikant_lookup.get(int(row['fabrikant_bk'])),
            'inkoop_type': row['inkoop_type'],
            'aantal': int(row['aantal']),
            'inkoopprijs_eenheid': None if pd.isna(row['inkoopprijs_eenheid']) else float(row['inkoopprijs_eenheid']),
            'inkoopbedrag': None if pd.isna(row['inkoopbedrag']) else float(row['inkoopbedrag'])
        })
    if inkoop_rows:
        pd.DataFrame(inkoop_rows).to_sql('fact_inkoop', con, if_exists='append', index=False)

    existing_onderhoud = set(pd.read_sql_query("SELECT onderhoud_bk FROM fact_onderhoud", con)['onderhoud_bk'].tolist())
    onderhoud_rows = []
    for _, row in staging['fact_onderhoud'].iterrows():
        if row['onderhoud_bk'] in existing_onderhoud:
            continue
        filiaal_bk = monteur_filiaal.get(row['monteur_bk'])
        onderhoud_rows.append({
            'onderhoud_bk': row['onderhoud_bk'],
            'date_key': int(row['date_key']),
            'monteur_key': monteur_lookup.get(row['monteur_bk']),
            'filiaal_key': filiaal_lookup.get(filiaal_bk),
            'product_key': product_lookup.get(row['product_bk']),
            'starttijd': row['starttijd'],
            'eindtijd': row['eindtijd'],
            'duur_uren': float(row['duur_uren']),
            'uurloon': float(monteur_uurloon.get(row['monteur_bk'])),
            'arbeidskosten': float(row['duur_uren']) * float(monteur_uurloon.get(row['monteur_bk']))
        })
    if onderhoud_rows:
        pd.DataFrame(onderhoud_rows).to_sql('fact_onderhoud', con, if_exists='append', index=False)

    con.commit()

def run_etl(load_date='2026-04-08'):
    staging = build_staging(WORK_SOURCES)
    con = connect(DWH_PATH)
    con.executescript(SCHEMA_SQL)
    sync_date_dim(con, staging['dim_date'])

    load_dim_type1(
        con, 'dim_filiaal', staging['dim_filiaal'], 'filiaalnr',
        {'filiaalnr': 'filiaal_bk', 'naam': 'naam', 'adres': 'adres', 'provincie': 'provincie'}
    )
    load_dim_type2(
        con, 'dim_klant', staging['dim_klant'], 'klantnr',
        ['naam', 'adres', 'woonplaats', 'geslacht', 'geboortedatum', 'leeftijd', 'leeftijdsklasse'],
        {'klantnr': 'klant_bk', 'naam': 'naam', 'adres': 'adres', 'woonplaats': 'woonplaats', 'geslacht': 'geslacht',
          'geboortedatum': 'geboortedatum', 'leeftijd': 'leeftijd', 'leeftijdsklasse': 'leeftijdsklasse'},
        load_date=load_date
    )
    load_dim_type2(
        con, 'dim_monteur', staging['dim_monteur'], 'monteurnr',
        ['naam', 'woonplaats', 'uurloon', 'looncategorie', 'filiaal'],
        {'monteurnr': 'monteur_bk', 'naam': 'naam', 'woonplaats': 'woonplaats', 'uurloon': 'uurloon',
          'looncategorie': 'looncategorie', 'filiaal': 'filiaal_bk'},
        load_date=load_date
    )
    load_dim_type1(
        con, 'dim_leverancier', staging['dim_leverancier'], 'leveranciernr',
        {'leveranciernr': 'leverancier_bk', 'naam': 'naam', 'adres': 'adres', 'woonplaats': 'woonplaats'}
    )
    load_dim_type1(
        con, 'dim_fabrikant', staging['dim_fabrikant'], 'fabrikantnr',
        {'fabrikantnr': 'fabrikant_bk', 'naam': 'naam', 'adres': 'adres', 'plaats': 'plaats'}
    )
    load_dim_type1(
        con, 'dim_product', staging['dim_product'], 'product_bk',
        {'product_bk': 'product_bk', 'product_type': 'product_type', 'source_domain': 'source_domain', 'natural_key': 'natural_key',
          'naam': 'naam', 'soort': 'soort', 'merk': 'merk', 'model_type': 'model_type', 'kleur': 'kleur',
          'standaardprijs': 'standaardprijs', 'inkoopprijs': 'inkoopprijs', 'prijssegment': 'prijssegment',
          'winstmarge_pct': 'winstmarge_pct', 'leverancier_bk': 'leverancier_bk', 'fabrikant_bk': 'fabrikant_bk'}
    )
    load_facts(con, staging)

    summary = {}
    for table in ['dim_date', 'dim_filiaal', 'dim_klant', 'dim_monteur', 'dim_leverancier', 'dim_fabrikant', 'dim_product',
                  'fact_verkoop', 'fact_inkoop', 'fact_onderhoud']:
        summary[table] = int(pd.read_sql_query(f"SELECT COUNT(*) AS aantal FROM {table}", con)['aantal'].iloc[0])
    con.close()
    return pd.DataFrame(summary.items(), columns=['tabel', 'aantal'])

def q(sql):
    con = connect(DWH_PATH)
    df = pd.read_sql_query(sql, con)
    con.close()
    return df


## 1. Werkbestanden resetten en leeg DWH aanmaken

In [2]:
reset_working_sources()
reset_dwh()
print('Working copies aangemaakt in:', WORK_DIR)
print('Leeg DWH aangemaakt op:', DWH_PATH)

Working copies aangemaakt in: /mnt/data/working_sources
Leeg DWH aangemaakt op: /mnt/data/BikeToDrive_DWH.db


## 2. Eerste ETL-run

In [3]:
run_etl(load_date='2026-04-08')

,tabel,aantal
0,dim_date,201
1,dim_filiaal,5
2,dim_klant,25
3,dim_monteur,15
4,dim_leverancier,5
5,dim_fabrikant,11
6,dim_product,203
7,fact_verkoop,250
8,fact_inkoop,150
9,fact_onderhoud,50


## 3. Controle: aantallen per tabel

In [4]:
q("""
SELECT 'dim_date' AS tabel, COUNT(*) AS aantal FROM dim_date
UNION ALL SELECT 'dim_filiaal', COUNT(*) FROM dim_filiaal
UNION ALL SELECT 'dim_klant', COUNT(*) FROM dim_klant
UNION ALL SELECT 'dim_monteur', COUNT(*) FROM dim_monteur
UNION ALL SELECT 'dim_leverancier', COUNT(*) FROM dim_leverancier
UNION ALL SELECT 'dim_fabrikant', COUNT(*) FROM dim_fabrikant
UNION ALL SELECT 'dim_product', COUNT(*) FROM dim_product
UNION ALL SELECT 'fact_verkoop', COUNT(*) FROM fact_verkoop
UNION ALL SELECT 'fact_inkoop', COUNT(*) FROM fact_inkoop
UNION ALL SELECT 'fact_onderhoud', COUNT(*) FROM fact_onderhoud
""")

,tabel,aantal
0,dim_date,201
1,dim_filiaal,5
2,dim_klant,25
3,dim_monteur,15
4,dim_leverancier,5
5,dim_fabrikant,11
6,dim_product,203
7,fact_verkoop,250
8,fact_inkoop,150
9,fact_onderhoud,50


## 4. Test SCD Type 1
We passen een product aan in het SDM. Daarna draaien we de ETL opnieuw. Verwachting: de bestaande rij in `dim_product` wordt overschreven.

In [5]:
q("""
SELECT product_key, product_bk, naam, standaardprijs, prijssegment
FROM dim_product
WHERE product_bk = 'ACCESSOIRE_VERKOOP_1'
""")

,product_key,product_bk,naam,standaardprijs,prijssegment
0,1,ACCESSOIRE_VERKOOP_1,LED voorlamp,14.99,Laag


In [6]:
con = connect(WORK_SOURCES['acc_sale'], foreign_keys=False)
con.execute("UPDATE Accessoire SET naam = ?, standaardprijs = ? WHERE accessoirenr = 1", ('LED voorlamp XL', 16.49))
con.commit()
con.close()

run_etl(load_date='2026-04-09')

,tabel,aantal
0,dim_date,201
1,dim_filiaal,5
2,dim_klant,25
3,dim_monteur,15
4,dim_leverancier,5
5,dim_fabrikant,11
6,dim_product,203
7,fact_verkoop,250
8,fact_inkoop,150
9,fact_onderhoud,50


In [7]:
q("""
SELECT product_key, product_bk, naam, standaardprijs, prijssegment
FROM dim_product
WHERE product_bk = 'ACCESSOIRE_VERKOOP_1'
""")

,product_key,product_bk,naam,standaardprijs,prijssegment
0,1,ACCESSOIRE_VERKOOP_1,LED voorlamp XL,16.49,Midden


## 5. Test SCD Type 2 op klant
We passen een klant aan in het SDM. Daarna draaien we de ETL opnieuw. Verwachting:
- oude rij blijft bestaan en krijgt `is_current = 0`
- nieuwe rij wordt toegevoegd met `is_current = 1`

In [8]:
q("""
SELECT klant_key, klant_bk, naam, adres, woonplaats, valid_from, valid_to, is_current
FROM dim_klant
WHERE klant_bk = 1
ORDER BY klant_key
""")

,klant_key,klant_bk,naam,adres,woonplaats,valid_from,valid_to,is_current
0,1,1,Jan Jansen,Kerkstraat 12,Amsterdam,2026-04-08,None,1


In [9]:
for db in [WORK_SOURCES['acc_sale'], WORK_SOURCES['bike_sale']]:
    con = connect(db, foreign_keys=False)
    con.execute("UPDATE Klant SET adres = ?, woonplaats = ? WHERE klantnr = 1", ('Nieuwe Markt 99', 'Utrecht'))
    con.commit()
    con.close()

run_etl(load_date='2026-04-10')

,tabel,aantal
0,dim_date,201
1,dim_filiaal,5
2,dim_klant,26
3,dim_monteur,15
4,dim_leverancier,5
5,dim_fabrikant,11
6,dim_product,203
7,fact_verkoop,250
8,fact_inkoop,150
9,fact_onderhoud,50


In [10]:
q("""
SELECT klant_key, klant_bk, naam, adres, woonplaats, valid_from, valid_to, is_current
FROM dim_klant
WHERE klant_bk = 1
ORDER BY klant_key
""")

,klant_key,klant_bk,naam,adres,woonplaats,valid_from,valid_to,is_current
0,1,1,Jan Jansen,Kerkstraat 12,Amsterdam,2026-04-08,2026-04-10,0
1,26,1,Jan Jansen,Nieuwe Markt 99,Utrecht,2026-04-10,None,1


## 6. Extra SCD Type 2 test op monteur
Ook `dim_monteur` is Type 2. We wijzigen het uurloon van monteur 1.

In [11]:
q("""
SELECT monteur_key, monteur_bk, naam, uurloon, looncategorie, filiaal_bk, valid_from, valid_to, is_current
FROM dim_monteur
WHERE monteur_bk = 1
ORDER BY monteur_key
""")

,monteur_key,monteur_bk,naam,uurloon,looncategorie,filiaal_bk,valid_from,valid_to,is_current
0,1,1,Tom van Dijk,19.5,Midden,1,2026-04-08,None,1


In [12]:
for db in [WORK_SOURCES['acc_sale'], WORK_SOURCES['bike_sale'], WORK_SOURCES['maint']]:
    con = connect(db, foreign_keys=False)
    con.execute("UPDATE Monteur SET uurloon = ? WHERE monteurnr = 1", (22.75,))
    con.commit()
    con.close()

run_etl(load_date='2026-04-11')

,tabel,aantal
0,dim_date,201
1,dim_filiaal,5
2,dim_klant,26
3,dim_monteur,16
4,dim_leverancier,5
5,dim_fabrikant,11
6,dim_product,203
7,fact_verkoop,250
8,fact_inkoop,150
9,fact_onderhoud,50


In [13]:
q("""
SELECT monteur_key, monteur_bk, naam, uurloon, looncategorie, filiaal_bk, valid_from, valid_to, is_current
FROM dim_monteur
WHERE monteur_bk = 1
ORDER BY monteur_key
""")

,monteur_key,monteur_bk,naam,uurloon,looncategorie,filiaal_bk,valid_from,valid_to,is_current
0,1,1,Tom van Dijk,19.50,Midden,1,2026-04-08,2026-04-11,0
1,16,1,Tom van Dijk,22.75,Hoog,1,2026-04-11,None,1


## 7. Nieuwe feitregel die verwijst naar gewijzigde business key
We voegen een nieuwe verkoop toe voor klant `klantnr = 1`. De ETL moet in `fact_verkoop` automatisch de **nieuwste surrogate key** uit `dim_klant` gebruiken.

In [14]:
con = connect(WORK_SOURCES['bike_sale'], foreign_keys=False)
max_id = pd.read_sql_query('SELECT MAX(fiets_verkoopnr) AS max_id FROM Fiets_Verkoop', con)['max_id'].iloc[0]
new_id = int(max_id) + 1
con.execute(
    "INSERT INTO Fiets_Verkoop (fiets_verkoopnr, datum, aantal, verkoopprijs, klant, fiets, monteur) VALUES (?, ?, ?, ?, ?, ?, ?)",
    (new_id, '2024-12-31', 1, 1499.99, 1, 1, 1)
)
con.commit()
con.close()

run_etl(load_date='2026-04-12')

,tabel,aantal
0,dim_date,201
1,dim_filiaal,5
2,dim_klant,26
3,dim_monteur,16
4,dim_leverancier,5
5,dim_fabrikant,11
6,dim_product,203
7,fact_verkoop,251
8,fact_inkoop,150
9,fact_onderhoud,50


In [15]:
q(f"""
SELECT fv.verkoop_bk,
       fv.klant_key,
       dk.klant_bk,
       dk.naam,
       dk.adres,
       dk.woonplaats,
       dk.is_current,
       fv.omzet_bedrag
FROM fact_verkoop fv
JOIN dim_klant dk ON dk.klant_key = fv.klant_key
WHERE fv.verkoop_bk = 'FIETS_{new_id}'
""")

,verkoop_bk,klant_key,klant_bk,naam,adres,woonplaats,is_current,omzet_bedrag
0,FIETS_151,26,1,Jan Jansen,Nieuwe Markt 99,Utrecht,1,1499.99


## 8. SQL DDL tonen
Deze cell laat alle `CREATE TABLE`-statements zien die ook in het losse `.sql`-bestand staan.

In [16]:
print(SCHEMA_SQL)

PRAGMA foreign_keys = ON;

CREATE TABLE IF NOT EXISTS dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date TEXT NOT NULL UNIQUE,
    day_of_month INTEGER NOT NULL,
    month_num INTEGER NOT NULL,
    month_name TEXT NOT NULL,
    quarter_num INTEGER NOT NULL,
    year_num INTEGER NOT NULL,
    week_num INTEGER NOT NULL,
    day_name TEXT NOT NULL,
    is_weekend INTEGER NOT NULL
);

CREATE TABLE IF NOT EXISTS dim_filiaal (
    filiaal_key INTEGER PRIMARY KEY AUTOINCREMENT,
    filiaal_bk INTEGER NOT NULL UNIQUE,
    naam TEXT,
    adres TEXT,
    provincie TEXT
);

CREATE TABLE IF NOT EXISTS dim_klant (
    klant_key INTEGER PRIMARY KEY AUTOINCREMENT,
    klant_bk INTEGER NOT NULL,
    naam TEXT,
    adres TEXT,
    woonplaats TEXT,
    geslacht TEXT,
    geboortedatum TEXT,
    leeftijd INTEGER,
    leeftijdsklasse TEXT,
    valid_from TEXT NOT NULL,
    valid_to TEXT,
    is_current INTEGER NOT NULL DEFAULT 1,
    UNIQUE (klant_bk, valid_from)
);

CREATE TABLE IF NOT EXISTS dim